In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [7]:
!pwd

In [9]:
slurmDF = pd.read_csv("../data/i/slurm_intermediate.csv")

In [ ]:
slurmDF.columns

In [ ]:
df = pd.read_parquet("supercloud_power/data/i/gpu/12734094968062.parquet")

In [ ]:
import glob

# Try to find the file in any subdirectory (since we're not sure which one contains the file)
csv_matches = glob.glob("supercloud_power/data/r/gpu/*/12734094968062-*.csv")
if not csv_matches:
    raise FileNotFoundError("Could not find the target CSV in any subdirectory!")
df2 = pd.read_csv(csv_matches[0]).rename(columns={"utilization_gpu_pct": "gpu_used_pct", "utilization_memory_pct": "memory_used_pct", "memory_used_MiB": "memory_used_MiB", "power_draw_W": "power_draw_W"})

In [43]:
import matplotlib.pyplot as plt
import numpy as np
import glob

job_id = "59256562408696"
df = pd.read_parquet(f"supercloud_power/data/i/gpu/{job_id}.parquet")

csv_matches = glob.glob(f"supercloud_power/data/r/gpu/*/{job_id}-*.csv")
if not csv_matches:
    raise FileNotFoundError("Could not find any target CSVs for this job in any subdirectory!")

metrics = [
    ('gpu_used_pct', 'GPU Used (%)'),
    ('memory_used_pct', 'Memory Used (%)'),
    ('memory_used_MiB', 'Memory Used (MiB)'),
    ('power_draw_W', 'Power Draw (W)'),
]

# Plot 1: Each metric from parquet vs its own timestamp
fig, axes = plt.subplots(4, 1, figsize=(18, 16), sharex=True)
for i, (col, ylabel) in enumerate(metrics):
    axes[i].plot(df['timestamp'], df[col], 'o-', color='C0', label='df (parquet, native)')
    axes[i].set_ylabel(ylabel)
    axes[i].set_title(f'{col} from Parquet vs Parquet Timestamps')
    axes[i].legend()
    axes[i].grid(True)
axes[-1].set_xlabel('Parquet Timestamp (seconds)')
plt.tight_layout()
plt.show()

# For multi-node jobs, plot each CSV file (one plot per file)
for csv_path in csv_matches:
    # Derive a label for the node (assume filename encodes node info right after job_id-)
    node_label = csv_path.split('/')[-1].split('-')[1] if '-' in csv_path.split('/')[-1] else csv_path.split('/')[-1]
    df2 = pd.read_csv(csv_path).rename(columns={
        "utilization_gpu_pct": "gpu_used_pct",
        "utilization_memory_pct": "memory_used_pct",
        "memory_used_MiB": "memory_used_MiB",
        "power_draw_W": "power_draw_W"
    })

    fig, axes = plt.subplots(4, 1, figsize=(18, 16), sharex=True)
    for i, (col, ylabel) in enumerate(metrics):
        axes[i].plot(df2['timestamp'], df2[col], 'o-', color='C1', label=f'csv ({node_label})')
        axes[i].set_ylabel(ylabel)
        axes[i].set_title(f'{col} from CSV vs CSV Timestamps ({node_label})')
        axes[i].legend()
        axes[i].grid(True)
    axes[-1].set_xlabel('CSV Timestamp (seconds)')
    plt.tight_layout()
    plt.show()

# 3. (Optional) Uniform grid resampling—ignore if not useful for your goal, but kept for easy toggling
from scipy.interpolate import interp1d

def resample_uniform(df, kind='linear', dt=0.1):
    t0 = df['timestamp'].min()
    t1 = df['timestamp'].max()
    grid = np.arange(t0, t1 + dt, dt)
    out = {'timestamp': grid}
    for col, _ in metrics:
        f = interp1d(df['timestamp'], df[col], kind=kind, bounds_error=False, fill_value="extrapolate", assume_sorted=True)
        out[col] = f(grid)
    return pd.DataFrame(out)

df_uniform = resample_uniform(df)

# Also resample all CSVs individually
df2_uniforms = []
for idx, csv_path in enumerate(csv_matches):
    df2 = pd.read_csv(csv_path).rename(columns={
        "utilization_gpu_pct": "gpu_used_pct",
        "utilization_memory_pct": "memory_used_pct",
        "memory_used_MiB": "memory_used_MiB",
        "power_draw_W": "power_draw_W"
    })
    df2_uniforms.append(resample_uniform(df2))

# 4. Print negative values for parquet, and for each csv
def print_neg_rows(df, name="df"):
    neg_mem = df[df['memory_used_MiB'] < 0]
    neg_pow = df[df['power_draw_W'] < 0]
    if not neg_mem.empty:
        print(f"Rows with negative memory_used_MiB in {name}:")
        display(neg_mem)
    if not neg_pow.empty:
        print(f"Rows with negative power_draw_W in {name}:")
        display(neg_pow)

print_neg_rows(df, "df (parquet)")
for csv_path in csv_matches:
    node_label = csv_path.split('/')[-1].split('-')[1] if '-' in csv_path.split('/')[-1] else csv_path.split('/')[-1]
    df2 = pd.read_csv(csv_path).rename(columns={
        "utilization_gpu_pct": "gpu_used_pct",
        "utilization_memory_pct": "memory_used_pct",
        "memory_used_MiB": "memory_used_MiB",
        "power_draw_W": "power_draw_W"
    })
    print_neg_rows(df2, f"df2 (csv, {node_label})")

In [1]:
# Why does the code break with "BrokenProcessPool"?
# Usually, at least one .parquet file is corrupted and reading it crashes the worker process (possibly even at the C++/native level).
# The solution is to scan all input files serially, identify bad ones, and then (carefully) delete them.
# THIS CODE marks and deletes those files for you.
# (Now rewritten to use ProcessPoolExecutor for speed, but to avoid pool death,
#   we catch and record errors in the subprocesses and clean up in serial.)

import os
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

gpu_dir = 'supercloud_power/data/i/gpu'
sampling_freq_counts = Counter()
bad_files = []

def process_file_parallel(fpath):
    try:
        df_gpu = pd.read_parquet(fpath, columns=["timestamp"])
        ts = df_gpu["timestamp"].sort_values().to_numpy()
        if len(ts) < 2:
            return (fpath, None, None)
        deltas = np.diff(ts)
        if np.median(deltas) > 50:
            deltas = deltas / 1000
        if np.median(deltas) > 50:  # Still large, maybe microseconds
            deltas = deltas / 1000
        median_delta = float(np.round(np.median(deltas), 3))
        return (fpath, median_delta, None)
    except Exception as e:
        # Don't print here, let main process handle printing
        return (fpath, None, str(e))

parquet_files = [
    os.path.join(gpu_dir, fname)
    for fname in os.listdir(gpu_dir)
    if fname.endswith(".parquet")
]

# Use ProcessPoolExecutor to speed up parity checks
results = []
with ProcessPoolExecutor() as executor:
    future_to_fpath = {executor.submit(process_file_parallel, fpath): fpath for fpath in parquet_files}
    for future in tqdm(as_completed(future_to_fpath), total=len(parquet_files), desc="Checking Parquet files (parallel)"):
        try:
            result = future.result()
            results.append(result)
        except Exception as exc:
            # This should rarely happen, but mark file as bad
            fpath = future_to_fpath[future]
            print(f"Unknown exception for {os.path.basename(fpath)}: {exc}")
            bad_files.append(fpath)

for fpath, median_delta, error_msg in results:
    if median_delta is not None:
        sampling_freq_counts[median_delta] += 1
    if error_msg is not None:
        print(f"Error reading {os.path.basename(fpath)}: {error_msg}")
        bad_files.append(fpath)

print("# Sampling frequency (seconds):")
for freq, count in sorted(sampling_freq_counts.items()):
    print(f"# {freq:.3f} - {count}")

print("\n# Will now delete files that failed to read (likely corrupted/truncated):")
for bf in bad_files:
    try:
        os.remove(bf)
        print(f"Deleted: {bf}")
    except Exception as e:
        print(f"FAILED to delete {bf}: {e}")

print(f"\n# Deleted {len(bad_files)} corrupted parquet files.")

In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

gpu_raw = Path('supercloud_power/data/r/gpu')
all_files = list(gpu_raw.glob("*/*.csv"))

def get_file_stats(f):
    try:
        df = pd.read_csv(f, usecols=["timestamp"])
        ts = df["timestamp"].sort_values().to_numpy()
        n = len(ts)
        if n == 0:
            return None
        duration = float(ts[-1]) - float(ts[0]) if n > 1 else 0.0
        delta_t = duration / n if n > 0 else float("nan")
        job_id, node_id = os.path.basename(f).replace(".csv", "", 1).split("-", 1)
        return (os.path.basename(f), job_id, node_id, n, duration, delta_t)
   
    except Exception as e:
        os.remove(f)
        print(f"Failed {os.path.basename(f)}: {e}")
        return None

data = []
n_workers = max(1, (os.cpu_count() or 4) - 2)
with ProcessPoolExecutor(max_workers=n_workers) as executor:
    futures = {executor.submit(get_file_stats, f): f for f in all_files}
    with tqdm(total=len(futures), desc=str(len(all_files))) as pbar:
        for fut in as_completed(futures):
            result = fut.result()
            if result is not None:
                data.append(result)
            pbar.update(1)

In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

gpu_intermediate = Path('supercloud_power/data/i/gpu')
all_ifiles = list(gpu_intermediate.glob("*.parquet"))

def get_file_stats(f):
    try:
        df = pd.read_parquet(f, columns=["timestamp"])
        ts = df["timestamp"].sort_values().to_numpy()
        n = len(ts)
        if n == 0:
            return None
        duration = float(ts[-1]) - float(ts[0]) if n > 1 else 0.0
        delta_t = duration / n if n > 0 else float("nan")
        return (os.path.basename(f).replace(".parquet", "", 1), n, duration, delta_t)
    except Exception as e:
        os.remove(f)
        print(f"Failed {os.path.basename(f)}: {e}")
        return None

idata = []
n_workers = max(1, (os.cpu_count() or 4) - 2)
with ProcessPoolExecutor(max_workers=n_workers) as executor:
    futures = {executor.submit(get_file_stats, f): f for f in all_ifiles}
    with tqdm(total=len(futures), desc=str(len(all_ifiles))) as pbar:
        for fut in as_completed(futures):
            result = fut.result()
            if result is not None:
                idata.append(result)
            pbar.update(1)

# 21318588217648, 22004448063108

93773: 100%|█████████████████████████████████████████████| 93773/93773 [00:38<00:00, 2416.70it/s]


In [ ]:
# Real Data 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df = pd.DataFrame(data, columns=["file", "job_id", "node_id", "length", "duration", "delta_t"])
df["duration"] = df["duration"]
df.sort_values("duration", ascending=True, inplace=True)
# df.to_csv("supercloud_power/data/i/gpu_stats.csv", index=False)

# df[df["duration"] < 21*24*60]["duration"].value_counts().sort_index().head(10)
df[df["duration"] < 5*60]["duration"].count()
df[df["duration"] > 864000]["duration"].count()
# df[df["job_id"] == "59256562408696"]


In [28]:
# 1000000//(24*3600)
388+110

In [3]:
# Resampled Data

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

idf = pd.DataFrame(idata, columns=["file", "length", "duration", "delta_t"])
idf.sort_values(["delta_t"], ascending=True, inplace=True)
# df.to_csv("supercloud_power/data/i/gpu_stats.csv", index=False)
# idf.head(10)
idf.head(10)
# 40499706	2177682.526	0.053770

# 367.0

,file,length,duration,delta_t
44345,49615403895497,2,0.103,0.051500
82184,8291125376632,2,0.103,0.051500
22910,30595155278131,2,0.103,0.051500
28314,35380532623625,3,0.206,0.068667
2936,12734094968062,137,14.008,0.102248
90040,89814553754738,290,29.767,0.102645
63503,66374220676769,295,30.282,0.102651
69381,71614951586439,308,31.621,0.102666
6999,1637274926233,330,33.887,0.102688
7906,17187534388660,332,34.093,0.102690


In [5]:
traces = pd.read_csv("supercloud_power/data/i/gpu_traces.csv")

In [7]:
traces.sort_values(["delta_t"], ascending=True, inplace=True)
traces.head(10)

,job_id,nodes_used,file_path,length,duration_sec,delta_t
32,1002326246320,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,21325,2196.372,0.103
33,10024220107549,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,9947,1024.438,0.103
34,10025051623498,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,17540,1806.517,0.103
35,10025196298270,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,4993,514.176,0.103
36,10025488456665,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,839095,86426.682,0.103
37,1002678716603,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,8593,884.976,0.103
38,10027066129418,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,159038,16380.811,0.103
39,10027615707142,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,393804,40561.709,0.103
40,10027711687130,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,158401,16315.200,0.103
41,10027861089148,1,/scratch/aa9360/supercloud_power/data/i/gpu/10...,419696,43228.585,0.103


In [25]:
import pandas as pd
pd.set_option('display.max_rows', None)
slurm = pd.read_parquet("supercloud_power/data/i/slurm_log.parquet")

(slurm["duration_sec"] // 3600).value_counts().rename_axis("duration_hour").reset_index(name="count")
# .rename(columns={"index": "duration_hour", "duration_sec": "count"})
# slurm.head()

,duration_hour,count
0,0.0,35278
1,1.0,8758
2,2.0,7995
3,3.0,6565
4,4.0,4858
5,5.0,2878
6,12.0,2079
7,6.0,1970
8,7.0,1769
9,8.0,1602


In [2]:
from pathlib import Path
import pandas as pd

gpu_intermediate = Path('supercloud_power/data/i/gpu')
all_ifiles = list(gpu_intermediate.glob("*.parquet"))

igpu = pd.read_parquet(all_ifiles[0])
igpu.columns


Index(['timestamp', 'gpu_used_pct', 'memory_used_pct', 'memory_used_MiB',
       'power_draw_W'],
      dtype='object')